# Versao 11 - Visao Geral Dos Dados

A `versao11` nasce como um experimento controlado sobre a `versao10`. A arquitetura continua a mesma. O que muda e o criterio de quais observacoes entram no treinamento.

## Hipotese central

Nas series de falha (`1..9`), o comeco frequentemente ainda descreve operacao normal. Isso cria uma contaminacao importante:

- a classe global da serie e de falha;
- mas boa parte dos pontos iniciais ainda representa comportamento normal;
- assim, o modelo pode acabar aprendendo uma fronteira de classe menos nitida.

A `versao11` tenta atacar esse ponto sem trocar a rede:

- remove do modelo as features com `null_pct = 100%`;
- no treino das classes `1..9`, descarta observacoes com `state` ausente ou normal e mantem apenas estados transientes e de falha;
- preserva a classe `0` completa, pois ela e justamente a referencia de operacao normal.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao11" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from versao11.pipeline_v11 import (
    ALL_NULL_FEATURE_COLUMNS,
    SOURCE_TYPE_MAPPING,
    SELECTED_FEATURE_COLUMNS,
    build_feature_selection_report,
    build_series_quality_report,
    discover_series_manifest,
)

DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
manifest = discover_series_manifest(DATASET_ROOT)
quality_report = build_series_quality_report(manifest)
feature_report = build_feature_selection_report()

print("Numero total de series analisadas:", len(manifest))
print("Numero de variaveis originais:", len(ALL_NULL_FEATURE_COLUMNS) + len(SELECTED_FEATURE_COLUMNS))
print("Numero de variaveis mantidas:", len(SELECTED_FEATURE_COLUMNS))
print("Features removidas:", ALL_NULL_FEATURE_COLUMNS)
print("Mapeamento de origem:", SOURCE_TYPE_MAPPING)
display(feature_report)
display(quality_report.head())

Numero total de series analisadas: 2228
Numero de variaveis originais: 27
Numero de variaveis mantidas: 18
Features removidas: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']
Mapeamento de origem: {'well': 0, 'simulated': 1, 'drawn': 2}


,column,null_pct,selected_for_modeling,selection_reason,column_type
0,ABER-CKGL,100.0,False,all_null_feature_removed,continuous
1,ABER-CKP,100.0,False,all_null_feature_removed,continuous
2,ESTADO-DHSV,NaN,True,kept_for_modeling,state
3,ESTADO-M1,NaN,True,kept_for_modeling,state
4,ESTADO-M2,NaN,True,kept_for_modeling,state
5,ESTADO-PXO,NaN,True,kept_for_modeling,state
6,ESTADO-SDV-GL,NaN,True,kept_for_modeling,state
7,ESTADO-SDV-P,NaN,True,kept_for_modeling,state
8,ESTADO-W1,NaN,True,kept_for_modeling,state
9,ESTADO-W2,NaN,True,kept_for_modeling,state


,file_path,series_id,class_label_int,source_type,n_rows_original,nan_ratio_selected_features,n_state_normal_rows,n_state_transient_rows,n_state_failure_rows,n_negative_state_rows,keep_for_v11,drop_reason
0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201010207,0,well,21474,0.0,17874,0,0,0,True,
1,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201060114,0,well,21527,0.0,17927,0,0,0,True,
2,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201110124,0,well,21517,0.0,17917,0,0,0,True,
3,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201160311,0,well,21410,0.0,17810,0,0,0,True,
4,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0__WELL-00001_20170201210228,0,well,21453,0.0,17853,0,0,0,True,


## Leitura academica

Esta versao nao testa "mais capacidade de rede". Ela testa uma pergunta mais especifica: o ganho ou a perda vem do que se mostra ao modelo no treino, e nao do aumento da arquitetura.